In [2]:
#========================
# SCADA Event_analytics_v1
#========================

In [3]:
# Import libraries

import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import sqlite3

In [4]:
# Connect to the database

sqliteConnection = sqlite3.connect('SCADA_Historian.db')

In [5]:
# Read data from sqlite3 to pandas dataframe

df = pd.read_sql_query("SELECT * FROM process_data", sqliteConnection, index_col="ID")
sqliteConnection.close()

In [6]:
print(df)
print(df.info())
print(df.describe())

               timestamp  temperature  pressure  temperature_change  \
ID                                                                    
1    2026-06-11 06:01:00           42      1.55                   0   
2    2026-06-11 06:02:00           42      1.48                   0   
3    2026-06-11 06:03:00           42      1.52                   0   
4    2026-06-11 06:04:00           42      1.43                   0   
5    2026-06-11 06:05:00           42      1.52                   0   
..                   ...          ...       ...                 ...   
362  2026-06-11 12:02:00           18      4.01                   0   
363  2026-06-11 12:03:00           18      4.03                   0   
364  2026-06-11 12:04:00           18      3.99                   0   
365  2026-06-11 12:05:00           16      3.52                  -2   
366  2026-06-11 12:06:00           16      3.28                   0   

     pressure_change  
ID                    
1               0.00  
2      

In [7]:
# Change dataime format

df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df)
print(df.info())
print(df.describe())

              timestamp  temperature  pressure  temperature_change  \
ID                                                                   
1   2026-06-11 06:01:00           42      1.55                   0   
2   2026-06-11 06:02:00           42      1.48                   0   
3   2026-06-11 06:03:00           42      1.52                   0   
4   2026-06-11 06:04:00           42      1.43                   0   
5   2026-06-11 06:05:00           42      1.52                   0   
..                  ...          ...       ...                 ...   
362 2026-06-11 12:02:00           18      4.01                   0   
363 2026-06-11 12:03:00           18      4.03                   0   
364 2026-06-11 12:04:00           18      3.99                   0   
365 2026-06-11 12:05:00           16      3.52                  -2   
366 2026-06-11 12:06:00           16      3.28                   0   

     pressure_change  
ID                    
1               0.00  
2              -0.07

In [8]:
event_30C = df[
    df["temperature_change"] == 23
]
print(event_30C)

              timestamp  temperature  pressure  temperature_change  \
ID                                                                   
32  2026-06-11 06:32:00          149      3.23                  23   
92  2026-06-11 07:32:00          102      0.29                  23   
153 2026-06-11 08:33:00           93     -0.29                  23   
275 2026-06-11 10:35:00          102     -1.47                  23   
276 2026-06-11 10:36:00          125     -1.07                  23   

     pressure_change  
ID                    
32              0.26  
92              0.06  
153             0.01  
275             0.92  
276             0.40  


In [9]:
event_30C_ID = event_30C.index.tolist()
print(event_30C_ID)
print(len(event_30C_ID))

[32, 92, 153, 275, 276]
5


In [10]:
ID_number=len((event_30C_ID))
df_event_filtered = pd.DataFrame()

event_filtered=[]

for i in range(ID_number):
    act_id = event_30C_ID[i]
    event_filtered_raw = df[(df.index >= act_id - 5) & (df.index <= act_id + 5)]
    event_filtered.append(event_filtered_raw)
    df_event_filtered = pd.concat(event_filtered, ignore_index=False)

print(df_event_filtered)

              timestamp  temperature  pressure  temperature_change  \
ID                                                                   
27  2026-06-11 06:27:00           87      1.42                   2   
28  2026-06-11 06:28:00           91      1.44                   4   
29  2026-06-11 06:29:00           93      1.51                   2   
30  2026-06-11 06:30:00           96      1.99                   3   
31  2026-06-11 06:31:00          126      2.97                  30   
32  2026-06-11 06:32:00          149      3.23                  23   
33  2026-06-11 06:33:00          175      3.36                  26   
34  2026-06-11 06:34:00          158      4.27                 -17   
35  2026-06-11 06:35:00          137      4.25                 -21   
36  2026-06-11 06:36:00          123      4.35                 -14   
37  2026-06-11 06:37:00          108      4.35                 -15   
87  2026-06-11 07:27:00           53     -0.68                   2   
88  2026-06-11 07:28

In [11]:
filtered_df = df[(df["temperature_change"] > 20) | (df["temperature_change"] < -11)].copy()
print(filtered_df)

              timestamp  temperature  pressure  temperature_change  \
ID                                                                   
31  2026-06-11 06:31:00          126      2.97                  30   
32  2026-06-11 06:32:00          149      3.23                  23   
33  2026-06-11 06:33:00          175      3.36                  26   
34  2026-06-11 06:34:00          158      4.27                 -17   
35  2026-06-11 06:35:00          137      4.25                 -21   
..                  ...          ...       ...                 ...   
342 2026-06-11 11:42:00           91      2.89                 -12   
343 2026-06-11 11:43:00           78      2.95                 -13   
344 2026-06-11 11:44:00           60      3.98                 -18   
345 2026-06-11 11:45:00           40      4.99                 -20   
346 2026-06-11 11:46:00           24      5.00                 -16   

     pressure_change  
ID                    
31              0.98  
32              0.26

In [12]:
filtered_df["cycle_id"] = (filtered_df.index != filtered_df.index.to_series().shift(1) + 1).cumsum()
print(filtered_df)

              timestamp  temperature  pressure  temperature_change  \
ID                                                                   
31  2026-06-11 06:31:00          126      2.97                  30   
32  2026-06-11 06:32:00          149      3.23                  23   
33  2026-06-11 06:33:00          175      3.36                  26   
34  2026-06-11 06:34:00          158      4.27                 -17   
35  2026-06-11 06:35:00          137      4.25                 -21   
..                  ...          ...       ...                 ...   
342 2026-06-11 11:42:00           91      2.89                 -12   
343 2026-06-11 11:43:00           78      2.95                 -13   
344 2026-06-11 11:44:00           60      3.98                 -18   
345 2026-06-11 11:45:00           40      4.99                 -20   
346 2026-06-11 11:46:00           24      5.00                 -16   

     pressure_change  cycle_id  
ID                              
31              0.98   

In [21]:
df_event_data = pd.DataFrame({"cycle_id", "start_id", "end_id", "duration", "max_temperature", "min_temperature" , "total_rise" ,"total_fall", "absolute_delta", "peak_temperature_change", "valley_temperature_change" })

In [22]:
df_reset = filtered_df.reset_index()

cycle_id=[]
start_id=[]
end_id=[]
duration=[]
max_temperature=[]
min_temperature=[]
total_rise=[]
total_fall=[]
absolute_delta=[]
net_change=[]

for cycle in df_reset["cycle_id"].unique():
    df_cycle = df_reset[df_reset["cycle_id"] == cycle]
    
    cycle_id.append(cycle)
    start_id.append(df_cycle['ID'].iloc[0]) 
    end_id.append(df_cycle['ID'].iloc[-1])
    duration.append((df_cycle['timestamp'].iloc[-1]-df_cycle['timestamp'].iloc[0]).total_seconds()/60)
    max_temperature.append(df_cycle['temperature'].max())
    min_temperature.append(df_cycle['temperature'].min())
    total_rise.append(df_cycle[df_cycle['temperature_change']>0]['temperature_change'].sum())
    total_fall.append(df_cycle[df_cycle['temperature_change']<0]['temperature_change'].sum())
    absolute_delta.append(df_cycle["temperature_change"].abs().sum())
    net_change.append((df_cycle[df_cycle['temperature_change']>0]['temperature_change'].sum())+(df_cycle[df_cycle['temperature_change']<0]['temperature_change'].sum()))


In [23]:
df_event_data = pd.DataFrame({
    'cycle_id': cycle_id,
    'start_id': start_id,
    'end_id': end_id,
    'duration': duration,
    'max_temperature': max_temperature,
    'min_temperature': min_temperature,
    'total_rise': total_rise,
    'total_fall': total_fall,
    'absolute_delta': absolute_delta,
    'net_change': net_change
})

In [24]:
print(cycle_id)
print(start_id)
print(end_id)
print(duration)
print(max_temperature)
print(min_temperature)
print(total_rise)
print(total_fall)
print(absolute_delta)
print(net_change)
print(df_event_data)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[31, 92, 97, 101, 153, 163, 214, 218, 275, 336]
[40, 95, 99, 102, 161, 163, 216, 224, 285, 346]
[9.0, 3.0, 2.0, 1.0, 8.0, 0.0, 2.0, 6.0, 10.0, 10.0]
[175, 157, 111, 53, 145, 11, 162, 133, 152, 144]
[56, 102, 85, 37, 36, 11, 107, 35, 5, 24]
[79, 78, 0, 0, 75, 0, 82, 0, 73, 78]
[-119, -19, -42, -37, -109, -14, 0, -116, -147, -120]
[198, 97, 42, 37, 184, 14, 82, 116, 220, 198]
[-40, 59, -42, -37, -34, -14, 82, -116, -74, -42]
   cycle_id  start_id  end_id  duration  max_temperature  min_temperature  \
0         1        31      40       9.0              175               56   
1         2        92      95       3.0              157              102   
2         3        97      99       2.0              111               85   
3         4       101     102       1.0               53               37   
4         5       153     161       8.0              145               36   
5         6       163     163       0.0               11               11   
6 

In [27]:
sqliteConnection = sqlite3.connect('Event_analysis.db')

cursor = sqliteConnection.cursor()

table_creation_query = """
    CREATE TABLE Event_data (
        cycle_id INTEGER PRIMARY KEY NOT NULL
        start_id INTEGER,
        end_id INTEGER,
        duration REAL,
        max_temperature INTEGER,
        min_temperature INTEGER,
        total_rise INTEGER,
        total_fall INTEGER,
        absolute_delta INTEGER,
        net_change INTEGER
    );
"""

df_event_data.to_sql('Event_data', sqliteConnection, if_exists="replace", index=False)

sqliteConnection.commit()
sqliteConnection.close()


This module performs detailed event and process-cycle analysis based on the observations made during the exploratory data analysis.

The identified significant temperature changes are treated as parts of larger process cycles. Each detected cycle receives a unique `cycle_id`, and its main characteristics are calculated.

The following cycle-level features are generated:
- start and end ID
- duration
- maximum and minimum temperature
- total temperature rise
- total temperature fall
- absolute temperature change
- net temperature change

The resulting cycle-level dataset is stored in `Event_analysis.db` in the `Event_data` table.

### Result
The raw Historian measurements have been transformed into structured, cycle-level analytical data that can be used for visualization and further analysis.

### Next Step
Visualize and analyze the calculated event characteristics in Power BI.